# 📊 Data Analysis — 10-Week Course
## Week 2: Data Collection & Sources

---
### Learning Objectives
- Identify primary vs secondary data sources
- Learn methods of data acquisition: APIs, web scraping, databases, surveys
- Import datasets from CSV, Excel, and JSON
- Connect to a SQL database and run queries from Python
- Evaluate data source quality and reliability

### Outline
1. Primary vs Secondary Data  
2. Common Data Sources  
3. Importing CSV & Excel Files  
4. Reading JSON & APIs  
5. SQL Databases with Python  
6. Data Quality Checklist  
7. Lab Exercises  
8. Assignment  

In [ ]:
import numpy as np
import pandas as pd
import sqlite3, json, os
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

---
## 1. Primary vs Secondary Data

| Feature | Primary Data | Secondary Data |
|---|---|---|
| **Definition** | Collected first-hand for your specific question | Already exists; collected by someone else |
| **Examples** | Surveys, experiments, interviews, sensors | Census, WHO reports, Kaggle datasets |
| **Cost** | High (time + money) | Low |
| **Relevance** | Perfectly tailored | May not fit exactly |
| **Timeliness** | Current | May be outdated |
| **Control** | Full control over quality | Quality depends on original collector |

### Rule of thumb
Always check for secondary data first. If it exists and is reliable, use it.  
Collect primary data only when no suitable secondary source exists.

---
## 2. Common Data Sources

### Public / Open Datasets
| Source | URL | Content |
|---|---|---|
| World Bank | data.worldbank.org | Development, economics |
| WHO | apps.who.int/gho/data | Global health |
| UN Data | data.un.org | Population, trade |
| Kaggle | kaggle.com/datasets | Diverse topics, competitions |
| UCI ML Repository | archive.ics.uci.edu | ML benchmark datasets |
| Our World in Data | ourworldindata.org | Long-run global trends |
| Africa Open Data | africaopendata.org | Africa-specific datasets |

### Data Acquisition Methods
| Method | Tool | Use Case |
|---|---|---|
| File download | `pd.read_csv()` | CSV, Excel, JSON files |
| REST API | `requests` library | Live data, services |
| SQL database | `sqlite3`, `SQLAlchemy` | Relational databases |
| Web scraping | `BeautifulSoup`, `Selenium` | HTML tables, pages |
| Cloud storage | `boto3`, `gcloud` | AWS S3, Google Cloud |

---
## 3. Importing CSV & Excel Files

In [ ]:
# Create sample files to work with
N = 54
countries = [
    "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
    "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
    "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
    "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
    "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
    "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
    "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
    "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
]
regions = (["West Africa"]*16+["East Africa"]*14+
           ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
df_base = pd.DataFrame({
    "country"           :countries,
    "region"            :regions,
    "life_expectancy"   :np.clip(np.random.normal(63,8,N),45,80),
    "infant_mortality"  :np.clip(np.random.exponential(35,N),5,120),
    "gdp_per_capita"    :np.clip(np.exp(np.random.normal(7.2,1.2,N)),300,15000),
    "vaccination_dtp3"  :np.clip(np.random.normal(78,18,N),20,99),
    "water_access"      :np.clip(np.random.normal(68,22,N),15,99),
})

# Save as CSV
df_base.to_csv("africa_health.csv", index=False)

# Save as Excel (multiple sheets)
with pd.ExcelWriter("africa_health.xlsx", engine="openpyxl") as writer:
    df_base.to_excel(writer, sheet_name="Health", index=False)
    df_base.groupby("region").mean(numeric_only=True).round(1)\
           .to_excel(writer, sheet_name="Regional Summary")

print("Sample files created: africa_health.csv, africa_health.xlsx")

In [ ]:
# ── Reading CSV ──────────────────────────────────────────────────────
df_csv = pd.read_csv("africa_health.csv")
print("CSV — shape:", df_csv.shape)
print(df_csv.head(3).to_string())

# Useful read_csv options
df_limited = pd.read_csv(
    "africa_health.csv",
    usecols  = ["country", "region", "life_expectancy", "gdp_per_capita"],
    nrows    = 10,
    dtype    = {"life_expectancy": float},
    encoding = "utf-8"
)
print("\nLimited read:", df_limited.shape)

In [ ]:
# ── Reading Excel ────────────────────────────────────────────────────
df_excel = pd.read_excel("africa_health.xlsx", sheet_name="Health")
print("Excel — sheet 'Health':", df_excel.shape)

# Read all sheets at once
all_sheets = pd.read_excel("africa_health.xlsx", sheet_name=None)
print("Sheets available:", list(all_sheets.keys()))

summary_sheet = all_sheets["Regional Summary"]
print("\nRegional Summary sheet:")
print(summary_sheet.to_string())

---
## 4. Reading JSON & Simulated API Responses

In [ ]:
# Create a JSON file (semi-structured data)
json_data = {
    "metadata": {
        "source"      : "Africa Development Database",
        "year"        : 2022,
        "last_updated": "2023-01-15"
    },
    "countries": [
        {"name": "Ghana",  "region": "West Africa",
         "indicators": {"life_expectancy": 64.1, "gdp_per_capita": 2363, "hdi": 0.632}},
        {"name": "Kenya",  "region": "East Africa",
         "indicators": {"life_expectancy": 66.7, "gdp_per_capita": 1838, "hdi": 0.601}},
        {"name": "Egypt",  "region": "North Africa",
         "indicators": {"life_expectancy": 72.0, "gdp_per_capita": 3907, "hdi": 0.731}},
        {"name": "Rwanda", "region": "East Africa",
         "indicators": {"life_expectancy": 69.6, "gdp_per_capita": 822,  "hdi": 0.534}},
    ]
}

with open("health_data.json", "w") as f:
    json.dump(json_data, f, indent=2)

# Read JSON back
with open("health_data.json") as f:
    loaded = json.load(f)

print("Source:", loaded["metadata"]["source"])
print("Year:  ", loaded["metadata"]["year"])

# Flatten nested JSON into a DataFrame
records = []
for c in loaded["countries"]:
    row = {"name": c["name"], "region": c["region"]}
    row.update(c["indicators"])
    records.append(row)

df_json = pd.DataFrame(records)
print("\nFlattened DataFrame:")
print(df_json)

In [ ]:
# Simulating an API call (no internet required)
# In real usage: import requests; r = requests.get(url, params={...}); df = pd.json_normalize(r.json())

def mock_api_call(country, year=2022):
    """Simulates a REST API response for a health database."""
    np.random.seed(hash(country) % 1000)
    return {
        "status"  : 200,
        "country" : country,
        "year"    : year,
        "data": {
            "life_expectancy"   : round(np.clip(np.random.normal(63,8),45,80), 1),
            "infant_mortality"  : round(np.clip(np.random.exponential(35),5,120), 1),
            "vaccination_rate"  : round(np.clip(np.random.normal(75,15),20,99), 1),
        }
    }

# "Call the API" for several countries
query_countries = ["Ghana", "Kenya", "Nigeria", "Egypt", "South Africa"]
api_records = []

for country in query_countries:
    response = mock_api_call(country)
    if response["status"] == 200:
        row = {"country": response["country"], "year": response["year"]}
        row.update(response["data"])
        api_records.append(row)

df_api = pd.DataFrame(api_records)
print("API results:")
print(df_api)

---
## 5. SQL Databases with Python

In [ ]:
# Create and populate a SQLite database
conn = sqlite3.connect("health_database.db")
conn.row_factory = sqlite3.Row
cur  = conn.cursor()

cur.executescript("""
    DROP TABLE IF EXISTS countries;
    DROP TABLE IF EXISTS indicators;

    CREATE TABLE countries (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        name   TEXT NOT NULL UNIQUE,
        region TEXT NOT NULL
    );

    CREATE TABLE indicators (
        id                 INTEGER PRIMARY KEY AUTOINCREMENT,
        country_id         INTEGER REFERENCES countries(id),
        year               INTEGER,
        life_expectancy    REAL,
        infant_mortality   REAL,
        gdp_per_capita     REAL,
        vaccination_dtp3   REAL,
        water_access       REAL
    );
""")

# Insert countries
for i, (country, region) in enumerate(zip(df_base["country"], df_base["region"])):
    cur.execute("INSERT INTO countries (name, region) VALUES (?,?)", (country, region))

# Insert indicators
for _, row in df_base.iterrows():
    cur.execute("SELECT id FROM countries WHERE name=?", (row["country"],))
    cid = cur.fetchone()[0]
    cur.execute("""
        INSERT INTO indicators (country_id, year, life_expectancy, infant_mortality,
                                gdp_per_capita, vaccination_dtp3, water_access)
        VALUES (?,2022,?,?,?,?,?)
    """, (cid, row["life_expectancy"], row["infant_mortality"],
          row["gdp_per_capita"], row["vaccination_dtp3"], row["water_access"]))

conn.commit()
print("Database created with", cur.execute("SELECT COUNT(*) FROM countries").fetchone()[0],
      "countries and", cur.execute("SELECT COUNT(*) FROM indicators").fetchone()[0], "indicator records.")

In [ ]:
# ── Read SQL into Pandas DataFrames ─────────────────────────────────

# Simple SELECT
df_sql = pd.read_sql("SELECT * FROM countries LIMIT 5", conn)
print("Countries table:")
print(df_sql)

# JOIN query
query = """
    SELECT c.name, c.region,
           i.life_expectancy, i.gdp_per_capita, i.vaccination_dtp3
    FROM   countries c
    JOIN   indicators i ON c.id = i.country_id
    WHERE  i.life_expectancy > 65
    ORDER  BY i.life_expectancy DESC
    LIMIT  10
"""
df_query = pd.read_sql(query, conn)
print("\nCountries with life expectancy > 65 years:")
print(df_query.to_string(index=False))

# Aggregate query
agg_query = """
    SELECT c.region,
           COUNT(*)       AS n_countries,
           ROUND(AVG(i.life_expectancy),1) AS mean_life_exp,
           ROUND(AVG(i.gdp_per_capita),0)  AS mean_gdp
    FROM   countries c
    JOIN   indicators i ON c.id = i.country_id
    GROUP  BY c.region
    ORDER  BY mean_life_exp DESC
"""
df_agg = pd.read_sql(agg_query, conn)
print("\nRegional aggregates:")
print(df_agg.to_string(index=False))

conn.close()

---
## 6. Data Quality Checklist

Before using any dataset, assess it against these dimensions:

| Dimension | Question to Ask | Red flags |
|---|---|---|
| **Accuracy** | Are the values correct? | Impossible values (age=200) |
| **Completeness** | Are all expected records present? | >20% missing values |
| **Consistency** | Are formats/units uniform? | Mix of USD and local currency |
| **Timeliness** | Is the data current enough? | Data from 10 years ago for recent question |
| **Validity** | Do values conform to rules? | Negative income, invalid email |
| **Uniqueness** | Are there duplicate records? | Same person appears twice |
| **Provenance** | Who collected it and how? | Unknown methodology |

In [ ]:
def data_quality_report(df):
    """Run a comprehensive data quality check."""
    print("=" * 60)
    print("DATA QUALITY REPORT")
    print("=" * 60)
    print(f"  Shape        : {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Duplicates   : {df.duplicated().sum()} rows")
    print(f"  Total NaN    : {df.isnull().sum().sum()} cells")
    print()

    print(f"  {'Column':<25} {'Type':<10} {'Missing':>8} {'%':>6} {'Unique':>8}")
    print("  " + "-" * 62)
    for col in df.columns:
        s = df[col]
        miss = s.isna().sum()
        print(f"  {col:<25} {str(s.dtype):<10} {miss:>8} {100*miss/len(s):>5.1f}% {s.nunique():>8}")

    # Range checks for numeric columns
    print("\n  NUMERIC RANGES")
    for col in df.select_dtypes(include=np.number).columns:
        s = df[col].dropna()
        print(f"  {col:<25}: [{s.min():.1f}, {s.max():.1f}]  mean={s.mean():.1f}")

df = pd.read_csv("africa_health.csv")
data_quality_report(df)

---
## 7. Lab Exercises

### Lab 1: Multi-Source Data Integration
Combine the three data sources created in this notebook (CSV, JSON, API mock) into a single DataFrame.  
Handle any column name mismatches. Identify which source has the most complete data.

In [ ]:
# Lab 1


### Lab 2: SQL Analysis
Re-open `health_database.db` and write queries to find:
1. The 5 countries with the lowest vaccination rate
2. Countries where GDP > 2000 AND life expectancy < 60
3. The region with the highest average water access

In [ ]:
# Lab 2


---
## 8. Assignment — Week 2

**Problem 1 (30 pts): Data Source Audit**  
Find **three different** public datasets on the same topic (e.g., education in Africa). Fill in a table with: source name, URL, format, year, number of records, key variables, and a quality score (1–5) for each dimension in Section 6.

**Problem 2 (40 pts): Multi-Format Loading**  
Create a pipeline that:
- Loads `africa_health.csv` into a DataFrame
- Writes it to a SQLite database with appropriate schema
- Queries the database to get the top 10 countries by health score (formula: `0.5×life_expectancy + 0.3×vaccination - 0.2×infant_mortality`)
- Exports the results to an Excel file

**Problem 3 (30 pts): API Simulation**  
Extend `mock_api_call()` to:
- Accept a list of countries and return all results in one call
- Add a `retry` mechanism that retries up to 3 times if status ≠ 200
- Add a `rate_limit_delay` parameter (sleep between calls)
- Handle the case where a country name is not found

In [ ]:
# Problem 2


In [ ]:
# Problem 3


---
## Summary — Week 2

| Concept | Tool / Function |
|---|---|
| Primary vs secondary | Conceptual framework |
| Read CSV | `pd.read_csv(file, usecols=, nrows=, dtype=)` |
| Read Excel | `pd.read_excel(file, sheet_name=)` |
| Read JSON | `json.load(f)` → `pd.DataFrame(records)` |
| REST API | `requests.get(url)` → `pd.json_normalize(r.json())` |
| SQL → DataFrame | `pd.read_sql(query, conn)` |
| Data quality | Accuracy, Completeness, Consistency, Timeliness, Validity |

**Next:** Week 3 — Data Cleaning & Preprocessing